# ASR Roundtrip: Are the TTS Words Recognized Correctly?

In [16]:
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec peft

## Setup Granite Speech


In [17]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

GRANITE_ID = "ibm-granite/granite-speech-5.0-2b"
granite_processor = AutoProcessor.from_pretrained(GRANITE_ID)
granite_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    GRANITE_ID, dtype=torch.bfloat16).to("cuda")
print(f"Granite loaded on {granite_model.device} with dtype {granite_model.dtype}")

def transcribe_granite(wav, prompt=None, hotwords=None):
    """Granite Speech: chat-template prompt with <|audio|>; keywords appended for biasing."""
    if prompt is None:
        prompt = "can you transcribe the speech into a written format?"
    if hotwords:
        prompt = f"{prompt} Keywords: {hotwords}"
    chat = [
        {"role": "system", "content": "You are Granite, developed by IBM. You are a helpful AI assistant."},
        {"role": "user", "content": f"<|audio|>{prompt}"},
    ]
    text = granite_processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = granite_processor(text, wav, device=granite_model.device, return_tensors="pt").to(granite_model.device)
    with torch.inference_mode():
        out = granite_model.generate(**inputs, max_new_tokens=400, do_sample=False, num_beams=1, use_cache=True)
    return granite_processor.tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:],
                                      skip_special_tokens=True).strip()


Loading weights:   0%|          | 0/954 [00:00<?, ?it/s]

Granite loaded on cuda:0 with dtype torch.bfloat16


## Load the Generated TTS Sentences

In [18]:
from pathlib import Path
import pandas as pd
import torchaudio

problem_words = pd.read_csv("problem_words.csv", sep="\t")

TTS_DIR = Path("generated/kitten")

tts_samples = []
for w in problem_words["word"].head(1000):
    wav_path = TTS_DIR / f"{w}.wav"
    if wav_path.exists():
        tts_samples.append({"word": w, "audio": wav_path.as_posix()})

print(f"{len(tts_samples)} TTS samples loaded")
tts_samples[:3]

13 TTS samples loaded


[{'word': 'cristiano ronaldo',
  'audio': 'generated/kitten/cristiano ronaldo.wav'},
 {'word': 'twenty four', 'audio': 'generated/kitten/twenty four.wav'},
 {'word': 'john wesley hardin',
  'audio': 'generated/kitten/john wesley hardin.wav'}]

## Roundtrip Inference

In [19]:
from tqdm.auto import tqdm

def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)
    waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
    return waveform.mean(dim=0)

rows = []
for sample in tqdm(tts_samples):
    wav = load_audio(sample["audio"])
    hyp_granite = transcribe_granite(wav)
    rows.append({**sample,
                 "granite_hypothesis": hyp_granite,
})

roundtrip = pd.DataFrame(rows)
roundtrip.head()

  0%|          | 0/13 [00:00<?, ?it/s]

,word,audio,granite_hypothesis
0,cristiano ronaldo,generated/kitten/cristiano ronaldo.wav,here is the word cristiano ronaldo the word cr...
1,twenty four,generated/kitten/twenty four.wav,here is the word 24 the word 24 is often mispr...
2,john wesley hardin,generated/kitten/john wesley hardin.wav,here is the word john wesley harden the word j...
3,suttontaylor,generated/kitten/suttontaylor.wav,here is the word santon tailor the word santon...
4,steve lane,generated/kitten/steve lane.wav,here is the word steve lane the word steve lan...


## Word Recognition Analysis

In [20]:
import re
import string

def normalize(text):
    text = str(text).translate(str.maketrans("", "", string.punctuation)).lower().strip()
    text = re.sub(r" +", " ", text).strip()
    return text

def word_recognized(word, hypothesis):
    """an entity counts as recognized if every of its words appears in the hypothesis"""
    hyp_words = normalize(hypothesis).split()
    return all(w in hyp_words for w in normalize(word).split())

for asr_col in ["granite_hypothesis"]:
    asr_name = asr_col.replace("_hypothesis", "")
    roundtrip[f"{asr_name}_recognized"] = [
        word_recognized(w, h) for w, h in zip(roundtrip["word"], roundtrip[asr_col])
    ]

roundtrip.head()

,word,audio,granite_hypothesis,granite_recognized
0,cristiano ronaldo,generated/kitten/cristiano ronaldo.wav,here is the word cristiano ronaldo the word cr...,True
1,twenty four,generated/kitten/twenty four.wav,here is the word 24 the word 24 is often mispr...,False
2,john wesley hardin,generated/kitten/john wesley hardin.wav,here is the word john wesley harden the word j...,False
3,suttontaylor,generated/kitten/suttontaylor.wav,here is the word santon tailor the word santon...,False
4,steve lane,generated/kitten/steve lane.wav,here is the word steve lane the word steve lan...,True


## Recognition Rate

In [21]:
recognition_rate = roundtrip["granite_recognized"].mean()
print(f"Granite recognized {recognition_rate:.0%} of the TTS words")

Granite recognized 46% of the TTS words


In [25]:
per_word = roundtrip.set_index("word")[["granite_hypothesis", "granite_recognized"]]
per_word

,granite_hypothesis,granite_recognized
word,,
cristiano ronaldo,here is the word cristiano ronaldo the word cr...,True
twenty four,here is the word 24 the word 24 is often mispr...,False
john wesley hardin,here is the word john wesley harden the word j...,False
suttontaylor,here is the word santon tailor the word santon...,False
steve lane,here is the word steve lane the word steve lan...,True
groves,here is the word groves the word groves is oft...,True
kris taylor,here is the word chris taylor the word chris t...,False
missus hazelton,here is the word missus hazelton the word miss...,True
sybal,here is the word sibel the word sibel is often...,False


## Qualitative Inspection

In [26]:
from IPython.display import Audio, display

for _, row in roundtrip.iterrows():
    if not row["granite_recognized"]:
        print("=" * 70)
        print("WORD :", row["word"])
        print("GRANITE :", row["granite_hypothesis"])
        display(Audio(row["audio"]))

WORD : twenty four
GRANITE : here is the word 24 the word 24 is often mispronounced


WORD : john wesley hardin
GRANITE : here is the word john wesley harden the word john wesley harden is often mispronounced


WORD : suttontaylor
GRANITE : here is the word santon tailor the word santon tailor is often mispronounced


WORD : kris taylor
GRANITE : here is the word chris taylor the word chris taylor is often mispronounced


WORD : sybal
GRANITE : here is the word sibel the word sibel is often mispronounced


WORD : nieaner
GRANITE : here is the word neener the word neener is often mispronounced


WORD : jane dimond
GRANITE : here is the word jane diamond the word jane diamond is often mispronounced


## Save the Roundtrip Results

In [ ]:
roundtrip.to_csv("roundtrip_results.tsv", sep="\t", index=False)
print("Saved roundtrip_results.tsv and roundtrip_summary.tsv")